# 9 · Model families

WarpRec ships 87 models, and every one of them is found, built, trained, saved
and loaded the same way. This guide shows that common shape and trains one model
of each family on the same split. It covers:

- the model registry: how the models are organised, how a name in a
  configuration becomes a class, and how its hyperparameters are validated;
- the constructor every model shares, and the difference between models that
  fit in their constructor and models that a Lightning trainer trains;
- one model per family (unpersonalized, collaborative closed-form and
  iterative, sequential, content-based, hybrid) evaluated together;
- the design pipeline, which does all of the above from one configuration;
- saving a model with `get_state()` and restoring it with `from_checkpoint`.

**Requirements:** `pip install warprec jupyter`. Runs in about two minutes on a
laptop CPU, most of it training BPR and SASRec twice. It reads MovieLens-100K through
`guide_data.movielens_100k()`, which downloads and converts it exactly as
guide 1 shows.

In [1]:
import inspect
import logging
import sys
import time
import warnings
from collections import defaultdict
from pathlib import Path

import lightning as L
import pandas as pd
import torch
from loguru import logger

sys.path.insert(0, "..")
from guide_data import movielens_100k

folder = movielens_100k()
RUNS = Path("../runs/09-model-families")
RUNS.mkdir(parents=True, exist_ok=True)

# Lightning reports the hardware it found and warns about loader workers; none
# of it is about WarpRec, so it is kept out of the outputs.
logging.getLogger("lightning.pytorch").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", module="lightning")
warnings.filterwarnings("ignore", module="torch")
warnings.filterwarnings("ignore", message='install "ipywidgets"')
warnings.filterwarnings("ignore", message="enable_nested_tensor")
logger.disable("warprec")

## The registry

A model class registers itself with a decorator,
`@model_registry.register(name="EASE")`, and the schema of its hyperparameters
registers in `params_registry` under the same name. Importing WarpRec fills
both. The families are the sub-packages of `warprec.recommenders`; the table
counts, for each, the models that fit in closed form and the ones trained
iteratively, and what the dataset must carry for them (read from the
`need_*` flags of their schemas).

In [2]:
from warprec.recommenders.base_recommender import IterativeRecommender
from warprec.utils.registry import model_registry, params_registry

NEEDS = {
    "need_side_information": "side information",
    "need_context": "contexts",
    "need_knowledge": "knowledge graph",
    "need_multimodal": "multimodal features",
}


def family_of(cls):
    parts = cls.__module__.split(".")[2:]
    if parts[0] == "collaborative_filtering_recommender":
        return f"collaborative / {parts[1].replace('_', '-')}"
    return parts[0].removesuffix("_recommender").replace("_", "-")


families = defaultdict(list)
for key in model_registry.list_registered():
    families[family_of(model_registry.get_class(key))].append(key)

rows = []
for family, keys in sorted(families.items()):
    classes = [model_registry.get_class(k) for k in keys]
    schemas = [
        params_registry.get_class(k)
        for k in keys
        if k in params_registry.list_registered()
    ]
    needs = {
        label for flag, label in NEEDS.items() for s in schemas if getattr(s, flag)
    }
    rows.append(
        {
            "family": family,
            "models": len(keys),
            "closed form": sum(
                not issubclass(c, IterativeRecommender) for c in classes
            ),
            "iterative": sum(issubclass(c, IterativeRecommender) for c in classes),
            "needs": ", ".join(sorted(needs)) or "-",
            "e.g.": ", ".join(sorted(c.__name__ for c in classes)[:3]),
        }
    )
table = pd.DataFrame(rows).set_index("family")
print(len(model_registry.list_registered()), "registered names")
table

88 registered names


,models,closed form,iterative,needs,e.g.
family,,,,,
collaborative / autoencoder,8,2,6,-,"CDAE, DiffRec, EASE"
collaborative / graph-based,22,1,21,-,"DGCF, EGCF, ESIGCF"
collaborative / knn,4,4,0,-,"ItemKNN, ItemKNNTD, UserKNN"
collaborative / latent-factor,7,4,3,-,"ADMMSlim, BPR, FISM"
collaborative / neural,3,0,3,-,"ConvNCF, NeuMF, TwoTower"
content-based,1,1,0,side information,VSM
context-aware,8,0,8,contexts,"AFM, DCN, DCNv2"
hybrid,4,4,0,side information,"AddEASE, AttributeItemKNN, AttributeUserKNN"
knowledge-aware,7,0,7,knowledge graph,"CKE, KGAT, KGCN"


Closed form or iterative is a property of the model, not of the family: iALS
alternates its least-squares steps inside the constructor, and STAN, a
sequential model, is a neighbourhood method with nothing to train.

The 88 names are the 87 models and `ProxyRecommender`, which scores a
recommendation file produced elsewhere (guide 14). It is the only name without a
hyperparameter schema.

Names are stored upper-cased, so a lookup ignores case. `get_class` returns the
class; `get` builds an instance from the arguments you pass. An unknown name
raises and lists every registered one.

In [3]:
print(model_registry.get_class("ease") is model_registry.get_class("EASE"))
try:
    model_registry.get_class("EASY")
except ValueError as error:
    print(str(error)[:110], "…")

True
'EASY' not found in Recommender registry. Available options: ['CDAE', 'EASE', 'ELSA', 'MACRIDVAE', 'MULTIDAE', …


A model's constructor does not validate its hyperparameters: a pipeline first
passes them through the schema in `params_registry`, which checks types and
ranges and turns every value into a search space. A single value becomes a
one-point grid; a list becomes a grid over its values, which only the train
pipeline explores (guide 10).

In [4]:
from pydantic import ValidationError

print(params_registry.get("EASE", l2=200.0).l2)
print(params_registry.get("EASE", l2=[50.0, 200.0]).l2)
for bad in ({"l2": -1.0}, {}):
    try:
        params_registry.get("EASE", **bad)
    except ValidationError as error:
        print(f"{bad}:", str(error).splitlines()[2].strip()[:90])

[<SearchSpace.GRID: 'grid'>, 200.0]
[<SearchSpace.GRID: 'grid'>, 50.0, 200.0]
{'l2': -1.0}: Value error, Values of l2 for EASE model must be > 0. Values received as input: [-1.0] [ty
{}: Field required [type=missing, input_value={}, input_type=dict]


## The data

Every model below trains on the same `Dataset`. VSM and AttributeItemKNN score
from item features, so the genres of `items.tsv` are written as a table of
indicators and read as side information (guide 4 covers side information).

In [5]:
items = pd.read_csv(folder / "items.tsv", sep="\t")
genres = pd.concat([items[["item_id"]], items.genres.str.get_dummies(sep="|")], axis=1)
genres.to_csv(RUNS / "genres.tsv", sep="\t", index=False)
print(genres.shape)
genres.iloc[:2, :8]

(1682, 20)


,item_id,Action,Adventure,Animation,Children's,Comedy,Crime,Documentary
0,1,0,0,1,1,1,0,0
1,2,1,1,0,0,0,0,0


The configuration below is the one the design pipeline runs at the end of the
guide. Until then only its `reader` and `splitter` are used, to build the
dataset, and its `models` section is where the hyperparameters of each model are
read from.

In [6]:
print(Path("configs/design.yml").read_text())

# One model per family, each hyperparameter with a single value: the design
# pipeline trains every model once, in this process, and prints its test results.
# Paths are relative to the guide's folder.
reader:
  loading_strategy: dataset
  data_type: transaction
  reading_method: local
  local_path: ../data/ml-100k/ratings.tsv
  rating_type: implicit
  # Genres as item side information, for VSM and AttributeItemKNN (guide 4).
  side:
    local_path: ../runs/09-model-families/genres.tsv
    sep: "\t"
    header: true
splitter:
  test_splitting:
    strategy: temporal_holdout
    ratio: 0.1
# Sizes and epochs are small so that the guide runs in minutes on a CPU; they
# are not tuned. num_workers: 0 loads the data in this process, which on a
# dataset this small is faster than starting worker processes.
models:
  Pop:
    optimization: {num_workers: 0}
  EASE:
    l2: 200.0
    optimization: {num_workers: 0}
  BPR:
    embedding_size: 32
    reg_weight: 0.0001
    batch_size: 1024
    epo

In [7]:
from warprec.common.initialize import initialize_datasets
from warprec.data.reader import ReaderFactory
from warprec.utils.callback import WarpRecCallback
from warprec.utils.config.config import load_design_configuration

config = load_design_configuration("configs/design.yml")
main, _, _ = initialize_datasets(
    ReaderFactory.get_reader(config), WarpRecCallback(), config
)
info = main.info()
print(
    f"{info['n_users']} users, {info['n_items']} items, {info['n_features']} features;",
    f"train {main.train_set.get_sparse().nnz}, test {main.eval_set.get_sparse().nnz}",
)

943 users, 1648 items, 19 features; train 89561, test 10401


## The constructor every model shares

Every model is built with the same keyword arguments:

- `params`: its hyperparameters, a plain dictionary;
- `info`: `dataset.info()`, the dimensions (`n_users`, `n_items`, and
  `n_features`, contexts, entities or modalities when the dataset has them);
- the dataset's entities: `interactions` (the training `Interactions`),
  `sessions` (the time-ordered `Sessions`), `transactions` (rows with their
  contexts), `knowledge`, `multimodal`;
- `seed`, which seeds Python, NumPy and PyTorch before the weights are drawn.

A model names the entities it needs in its signature and swallows the rest in
`**kwargs`, so a caller can hand every model everything. That is what the
pipelines do, and what `build` below does:

In [8]:
entities = {
    "interactions": main.train_set,
    "sessions": main.train_session,
    "transactions": main.train_transactions,
    "knowledge": main.knowledge,
    "multimodal": main.multimodal,
}


def build(name, **params):
    """Build a registered model the way the pipelines do."""
    return model_registry.get(
        name,
        params=params or config.models[name],
        info=main.info(),
        seed=42,
        **entities,
    )


def required(name):
    """The constructor arguments a model cannot do without, besides params and info."""
    signature = inspect.signature(model_registry.get_class(name).__init__)
    return [
        p
        for p, v in signature.parameters.items()
        if p not in ("self", "params", "info")
        and v.default is inspect.Parameter.empty
        and v.kind not in (v.VAR_POSITIONAL, v.VAR_KEYWORD)
    ]


pd.DataFrame(
    {
        name: {
            "also requires": required(name),
            "iterative": issubclass(
                model_registry.get_class(name), IterativeRecommender
            ),
        }
        for name in config.models
    }
).T

,also requires,iterative
Pop,[interactions],False
EASE,[interactions],False
BPR,[],True
SASRec,[],True
VSM,[interactions],False
AttributeItemKNN,[interactions],False


A model whose constructor derives something from the training data (a
closed-form solution, or the user-item graph LightGCN propagates over) requires
`interactions`; the others are built from `params` and `info` alone and see the
data only through their training loader. That split matters again when a model
is saved.

### Closed-form models fit in the constructor

EASE, ItemKNN, Pop, VSM and the other closed-form models compute their whole
solution in `__init__`: once built, the model can score.

In [9]:
start = time.time()
ease = build("EASE")
fit_time = {"EASE": time.time() - start}
print(f"built and fitted in {fit_time['EASE']:.2f}s")
print("item-item weights:", ease.item_similarity.shape)
ease.predict(torch.tensor([0, 1]))[:, :5]

built and fitted in 0.10s
item-item weights: (1648, 1648)


tensor([[ 0.8702,  0.7551,  0.5382,  0.8850,  0.1966],
        [ 0.4850, -0.0185,  0.0767,  0.0433,  0.0089]], dtype=torch.float64)

`get_params()` returns the hyperparameters the model actually read: the keys of
`params` that the class declares. Anything else in the dictionary, such as the
`optimization` block of the configuration, is ignored by the model. `name_param`
turns those parameters into a deterministic name, which the pipelines use for
the files a model writes.

In [10]:
for model in (ease, build("EASE", l2=200.0), build("EASE", l2=50.0)):
    print(model.get_params(), "->", model.name_param)

{'l2': 200.0} -> EASE_WiseFoxOfForce
{'l2': 200.0} -> EASE_WiseFoxOfForce
{'l2': 50.0} -> EASE_ElatedFuriousOyster


The same hyperparameters always give the same name and different ones a
different name. The seed is not a hyperparameter, so it does not enter the name.

### Iterative models are built untrained

The other models subclass `IterativeRecommender`, which is both a WarpRec
`Recommender` and a Lightning `LightningModule`. They declare `epochs` and
`learning_rate` among their hyperparameters, and their constructor only draws the
initial weights. Each knows which loader it trains on: `get_dataloader` picks it
and passes the model's own `batch_size` (guide 8).

In [11]:
bpr = build("BPR")
loader = bpr.get_dataloader(main.train_set, main.train_session)
print(isinstance(bpr, L.LightningModule), bpr.epochs, bpr.learning_rate)
print(type(loader.dataset).__name__, "| batches per epoch:", len(loader))

True 10 0.005
ContrastiveDataset | batches per epoch: 88


### Training through Lightning

The design and train pipelines train an iterative model in four steps:

1. `model.set_optimization_parameters(optimizer, lr_scheduler)` from the
   `optimization` block (Adam with no scheduler when it sets neither);
2. `model.get_dataloader(...)` for the training data;
3. a `lightning.Trainer` with `max_epochs=model.epochs`, the precision and
   gradient-clipping arguments `lightning_runtime` derives from `optimization`,
   and a `WarpRecLightningIntegrationCallback`, which evaluates the model after
   each epoch and applies early stopping;
4. `trainer.fit(model, train_loader, val_loader)`.

The function below is that, without the per-epoch evaluation: no callback and no
validation loader. The parameters are validated through `params_registry` first,
as a pipeline does, to read the `optimization` block.

In [12]:
from warprec.recommenders.trainer.runtime import lightning_runtime


def train(model):
    """Train an iterative model as the pipelines do, without per-epoch evaluation."""
    optimization = params_registry.get(
        model.name, **config.models[model.name]
    ).optimization
    model.set_optimization_parameters(optimization.optimizer, optimization.lr_scheduler)
    trainer = L.Trainer(
        max_epochs=model.epochs,
        accelerator="cpu",
        **lightning_runtime(optimization, "cpu"),
        logger=False,
        enable_checkpointing=False,
        enable_progress_bar=False,
        enable_model_summary=False,
    )
    start = time.time()
    trainer.fit(
        model,
        train_dataloaders=model.get_dataloader(main.train_set, main.train_session),
    )
    return time.time() - start, trainer.callback_metrics["loss"].item()

To see what training changes, the untrained BPR is scored first. The `Evaluator`
is built from the metric names, the cut-offs and the training matrix (whose
items it removes from every ranking); guide 12 covers it in full. User-averaged
metrics come back per user, and are averaged here. One evaluator scores every
model of this guide, as in the pipelines.

In [13]:
from warprec.evaluation import Evaluator

evaluator = Evaluator(["nDCG", "Recall"], [10], train_set=main.train_set.get_sparse())
test_loader = main.get_evaluation_dataloader()


def evaluate(model):
    evaluator.evaluate(model, test_loader, strategy="full", dataset=main)
    return {
        f"{metric}@10": round(value.nanmean().item(), 4)
        for metric, value in evaluator.compute_results()[10].items()
    }


before = evaluate(bpr)
seconds, loss = train(bpr)
print(f"untrained: {before}")
print(
    f"trained:   {evaluate(bpr)}  ({bpr.epochs} epochs in {seconds:.0f}s, final loss {loss:.3f})"
)

untrained: {'nDCG@10': 0.0092, 'Recall@10': 0.0066}
trained:   {'nDCG@10': 0.1223, 'Recall@10': 0.1064}  (10 epochs in 7s, final loss 0.149)


## One model per family

The four remaining models are built the same way, and SASRec, the other
iterative one, is trained with the same function. The sequential model needs
nothing different from the caller: it trains on the loader its `get_dataloader`
returns, and the evaluator hands it each user's most recent training items.

In [14]:
fit_time["BPR"] = seconds
models = {"EASE": ease, "BPR": bpr}
for name in ["Pop", "SASRec", "VSM", "AttributeItemKNN"]:
    start = time.time()
    models[name] = build(name)
    fit_time[name] = time.time() - start
    if isinstance(models[name], IterativeRecommender):
        fit_time[name], _ = train(models[name])

family = {
    "Pop": "unpersonalized",
    "EASE": "collaborative, closed form",
    "BPR": "collaborative, iterative",
    "SASRec": "sequential",
    "VSM": "content-based",
    "AttributeItemKNN": "hybrid",
}
results = pd.DataFrame(
    {
        name: {
            "family": family[name],
            "fit/train (s)": round(fit_time[name], 2),
            **evaluate(models[name]),
        }
        for name in family
    }
).T
results

,family,fit/train (s),nDCG@10,Recall@10
Pop,unpersonalized,0.0,0.0854,0.0723
EASE,"collaborative, closed form",0.1,0.1542,0.1391
BPR,"collaborative, iterative",7.19,0.1223,0.1064
SASRec,sequential,13.78,0.1227,0.1098
VSM,content-based,0.01,0.0176,0.0158
AttributeItemKNN,hybrid,0.0,0.0133,0.0126


The numbers rank the families on this split only, with untuned hyperparameters.
What they show is that one evaluator compares every kind of model on the same
test set. The two models that read only the 19 genres score well below
popularity: a genre says little about which of a user's films come next.

The context-aware, knowledge-aware and multimodal families are built and trained
the same way; what differs is the dataset, which must carry contexts, a knowledge
graph or item features. Guides 5, 6 and 7 build those datasets and train a
model of each family on them.

## The design pipeline

`design_pipeline(path)` runs everything above from the configuration: it builds
the dataset, then for each entry of `models` builds the model with every entity,
trains it if it is iterative and prints its test results. It needs no Ray and
runs no hyperparameter search: it trains in this process and writes nothing to
disk.

Each hyperparameter must have a single value. A list is a search space, which
the design pipeline does not explore: that is the train pipeline's job
(guide 10), with the same `models` section.

Its trainer has the integration callback and the evaluation loader, so after
every epoch of an iterative model it evaluates on the **test** split and shows
the result in the progress bar. That is useful to watch a model train, not to
choose an epoch. The log is on for this run; Lightning adds a summary of each
iterative model's modules.

In [15]:
from warprec.pipelines import design_pipeline

logger.enable("warprec")
design_pipeline("configs/design.yml")
logger.disable("warprec")

09-10-2026 12:33:42 - WarpRec - 📝 Starting the Design Pipeline.


09-10-2026 12:33:42 - WarpRec - 📝 Reading design configuration file in: configs/design.yml


09-10-2026 12:33:42 - WarpRec - 📝 Reading process completed correctly.


09-10-2026 12:33:42 - WarpRec - 📝 Starting test splitting process with temporal_holdout splitting strategy.


09-10-2026 12:33:42 - WarpRec - 📝 Test splitting completed in : 0.00s


...

09-10-2026 12:34:05 - WarpRec - 📝 | Cutoff   |      nDCG |    Recall |


09-10-2026 12:34:05 - WarpRec - 📝 +==========+===========+===========+


09-10-2026 12:34:05 - WarpRec - 📝 | Top@10   | 0.0133134 | 0.0125513 |


09-10-2026 12:34:05 - WarpRec - 📝 +----------+-----------+-----------+


09-10-2026 12:34:05 - WarpRec - ✅ Design pipeline executed successfully. WarpRec is shutting down.


Every model except SASRec gives the same results as through the Python API: same
split, same seed, same loaders. SASRec differs slightly because the pipeline
evaluates it between epochs, which changes the random stream its dropout draws
from, so the trained weights are not the same.

## Saving and loading

`get_state()` returns everything needed to rebuild a model: its name, its
parameters, `info`, the PyTorch `state_dict`, and the model's public attributes
as `artifacts`. The pipelines save this dictionary with `torch.save` when a
model sets `meta.save_model: true` (guide 10), and serving loads it (guide 17).

In [16]:
state = models["BPR"].get_state()
sorted(state)

['artifacts',
 'format_version',
 'info',
 'name',
 'params',
 'state_dict',
 'warprec_version']

`from_checkpoint` on the model's class restores it, by one of three routes:

- a closed-form model whose constructor needs `interactions` gets its saved
  attributes back, without refitting and without the data;
- an iterative model built from `params` and `info` alone is constructed again
  and its `state_dict` loaded;
- an iterative model whose constructor needs `interactions` (LightGCN, for its
  graph) is saved whole, under `module`, and returned as it is.

Every model of this guide goes through a file and back, and the restored model
scores the same users exactly as the original:

In [17]:
from warprec.recommenders.base_recommender import SequentialRecommenderUtils

users = torch.arange(main.get_dims()[0])


def scores(model):
    kwargs = {"user_indices": users}
    if isinstance(model, SequentialRecommenderUtils):
        seq, length = main.train_session.get_user_history_sequences(
            users.tolist(), model.max_seq_len
        )
        kwargs |= {"user_seq": seq, "seq_len": length}
    model.eval()
    with torch.inference_mode():
        return model.predict(**kwargs)


rows = {}
for name, model in models.items():
    path = RUNS / f"{model.name_param}.pth"
    torch.save(model.get_state(), path)
    checkpoint = torch.load(path, weights_only=False)
    restored = type(model).from_checkpoint(checkpoint)
    rows[name] = {
        "file": path.name,
        "route": "module"
        if "module" in checkpoint
        else "rebuilt"
        if isinstance(model, IterativeRecommender)
        else "artifacts",
        "identical scores": torch.equal(scores(model), scores(restored)),
    }
pd.DataFrame(rows).T

,file,route,identical scores
EASE,EASE_WiseFoxOfForce.pth,artifacts,True
BPR,BPR_NippyValiantNewt.pth,rebuilt,True
Pop,Pop_MiniatureRuggedSerpent.pth,artifacts,True
SASRec,SASRec_LivelyCleverGoat.pth,rebuilt,True
VSM,VSM_MuscularLivelyManticore.pth,artifacts,True
AttributeItemKNN,AttributeItemKNN_BeneficialAdmirableBonobo.pth,artifacts,True


## Summary

| key | what it does |
|---|---|
| `models.<Name>` | one entry per model, by its registered name (case does not matter) |
| `models.<Name>.<hyperparameter>` | a single value for the design pipeline; a list is a search space for the train pipeline |
| `models.<Name>.optimization.num_workers` | loader worker processes for training and evaluation; `0` loads in the main process |
| `models.<Name>.optimization.optimizer`, `lr_scheduler`, `precision`, `gradient_clip` | how an iterative model is optimised (guide 10) |
| `models.<Name>.meta.save_model` | write `get_state()` to `<name_param>.pth` (train pipeline) |
| `reader.side` | item features, needed by the content-based and hybrid models |

The [Models configuration](https://warprec.readthedocs.io/en/latest/configuration/models/)
page describes the per-model sections, the [Design pipeline](https://warprec.readthedocs.io/en/latest/pipelines/design/)
page the pipeline, and the [Recommenders](https://warprec.readthedocs.io/en/latest/recommenders/)
section every model and its hyperparameters.